In [ ]:
# Cell 1: Imports & Configuration
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
from datetime import datetime

# Jupyter notebook configuration
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from validation.create_embedding_plots import create_embedding_plots
from validation.data_loading import setup_data_with_error_handling

# LearnM8 imports
from learnm8.oracles import CSVOracle
from learnm8.core.data_manager import DataManager
from learnm8.learners.ensemble import RFEnsemble
from learnm8.acquisition import TSNEKMeansAcquisition
from learnm8.utils.data_loaders import load_benchmark_data

In [ ]:
# Cell 7: Step 2 - Load Dataset
print("📊 Step 2: Loading dataset...")
target = "FEN1"
target_path = "/home/tony/LearnM8/data/FEN1_5fv7_scoring_and_consensus_maxAL_with_activity.csv"
target_column = "CHEMPLP"


try:
    compound_pool, ground_truth = load_benchmark_data(str(target_path), target_column)
    print(f"Loaded {len(compound_pool)} compounds")
    print(f"Target column: {target_column}")
    print(f"Score range: {ground_truth[target_column].min():.2f} to {ground_truth[target_column].max():.2f}")
    
except Exception as e:
    print(f"Failed to load dataset: {e}")
    print(f"\n❌ Error loading dataset: {e}")

In [ ]:
# Cell 9: Step 4 - DataManager Setup
print("⚙️  Step 4: Setting up data manager with error handling...")

# Initialize DataManager
data_manager = DataManager(results_dir=str(Path("./")), featurizer='morgan')

# Test and setup data with error handling
compound_pool = setup_data_with_error_handling(compound_pool, data_manager)

In [ ]:
# Cell 5: Step 6 - Initial Training
print("🎯 Step 6: Performing initial training...")

learner = RFEnsemble(n_estimators=3, random_states=[42, 43, 44])

# Determine initial training size
initial_size = min(100, len(compound_pool) // 10)  # Adaptive initial size
np.random.seed(42)
initial_indices = np.random.choice(len(compound_pool), size=initial_size, replace=False)

# Split data
labeled_compounds = compound_pool.iloc[initial_indices].copy()
unlabeled_compounds = compound_pool.drop(compound_pool.index[initial_indices]).copy()

# Create oracle and measure labeled compounds
oracle = CSVOracle(csv_path=str(target_path))
labeled_compounds = oracle.measure(labeled_compounds, [target_column])

# Train the model
learner.train(labeled_compounds, target_column, data_manager)
print(f"Initial training completed: {len(labeled_compounds)} labeled, {len(unlabeled_compounds)} unlabeled")

In [ ]:
# Cell 6: Step 7 - TSNE KMeans Acquisition
print("🎯 Step 7: Running TSNE KMeans acquisition...")

# Set acquisition parameters
batch_fraction = 0.01
batch_size = max(1, int(batch_fraction * len(unlabeled_compounds)))
print(f"Batch size: {batch_size}")

print(f"\n🎯 Acquisition Parameters:")
print(f"  • Batch fraction: {batch_fraction*100:.1f}%")
print(f"  • Batch size: {batch_size:,}")
print(f"  • Unlabeled pool: {len(unlabeled_compounds):,}")

# Get predictions for unlabeled compounds
print("Getting predictions for unlabeled compounds...")
predictions, uncertainties = learner.predict(unlabeled_compounds, data_manager)
ground_truth_unlabeled = oracle.measure(unlabeled_compounds, [target_column])[target_column].values

# Prepare unlabeled compounds with predictions
unlabeled_with_predictions = unlabeled_compounds.copy()
unlabeled_with_predictions['prediction'] = predictions
if uncertainties is not None:
	unlabeled_with_predictions['uncertainty'] = uncertainties
else:
	unlabeled_with_predictions['uncertainty'] = np.zeros(len(predictions))

# Get TSNE KMeans acquisition function
print("Initializing TSNE KMeans acquisition function...")
init_start_time = datetime.now()
tsne_kmeans_acquisition = TSNEKMeansAcquisition(
	data_manager=data_manager, 
	n_components=2, 
)
init_time = datetime.now() - init_start_time
print(f"Initialization time: {init_time.total_seconds():.2f} seconds")

# Perform acquisition using TSNE KMeans
print("Performing TSNE KMeans compound selection...")
selection_start_time = datetime.now()
kmeans_selection = tsne_kmeans_acquisition.select(compounds=unlabeled_with_predictions, n_select=batch_size)
kmeans_indices = kmeans_selection.index.values
selection_time = datetime.now() - selection_start_time
total_kmeans_time = init_time + selection_time
print(f"Selection time: {selection_time.total_seconds():.2f} seconds")

print(f"TSNE KMeans selected {len(kmeans_indices)} compounds")

In [ ]:
# Cell 7: Step 8 - Generate Embeddings & Visualizations
print("📊 Step 8: Generating visualizations...")

from sklearn.manifold import TSNE

plots_dir = Path(f"./plots_{target}")

# Generate TSNE embeddings from molecular fingerprints
print("Generating TSNE embeddings from molecular fingerprints")
features = data_manager.get_features(
    compound_ids=unlabeled_compounds['ID'].tolist(),
    smiles_list=unlabeled_compounds['SMILES'].tolist(),
    featurizer_type='morgan'
)

# Generate TSNE embeddings
tsne = TSNE(n_components=2, random_state=42)
features_array = features.toarray() if hasattr(features, 'toarray') else features
embeddings = tsne.fit_transform(features_array)

# Get cluster labels from the acquisition function
kmeans_labels = tsne_kmeans_acquisition.cluster_labels

# Convert selected indices to unlabeled dataset indices
unlabeled_positions = {idx: pos for pos, idx in enumerate(unlabeled_compounds.index)}
kmeans_selected_positions = [unlabeled_positions[idx] for idx in kmeans_indices if idx in unlabeled_positions]

# Create visualization for KMeans clustering
create_embedding_plots(embeddings, kmeans_labels, kmeans_selected_positions, f"TSNE KMeans - {total_kmeans_time.total_seconds():.2f}s", plots_dir)

In [ ]:
import shutil
# Clean up temporary files
shutil.rmtree(Path(data_manager.results_dir / ".cache/"), ignore_errors=True)